# CelebA: current CASIA MP / AM / OP attack comparison

Run the cells in order on a **Tesla T4 GPU**. This notebook reruns the
experiment from raw images and metadata in the repository. It prepares
its own cohort, RGB64 image cache, public SVD feature mapping, and frozen
protocol context. It requires no previous study, feature cache, checkpoint,
result bundle, or private Google Drive folder.

The comparison fixes p = 10 and independently samples 500 anchor-noise scales from Uniform(0, 0.08). MP, AM, and OP use exactly the same noisy uploads for each draw. There is no bin calibration, acceptance filter, or utility training. A separate zero-noise diagnostic is excluded from the 500 observations, mean summaries, and fitted curves.

Both datasets use h = 400, r = 1,600 centered orthonormal anchors,
participant-specific GDP transformations, spectral-only alignment, and
the frozen CASIA-WebFace InceptionResnetV1 identity auditor. The full
utility study uses the original CNN, 15 epochs, Adam at 1e-3, batch size
256, and selection by validation balanced accuracy.

The raw datasets are assumed to have been downloaded legally already.
Place their archives and metadata in `data/raw/celeba/` **inside the
repository**, following the repository's documented raw-data layout.
A fresh Colab clone copies code, not ignored local datasets: upload/copy
those raw inputs to the clone before the preparation cell. The notebook
does not connect Google Drive or change a running experiment.

Plan for a long run and substantial CPU RAM and local disk in addition
to the T4's 16 GB device memory. Use a high-RAM runtime when possible (24–32 GB host RAM is a useful planning range), with about 20 GiB free disk as a planning budget for preparation, feature rows, checkpoints, and results. The decoded and projected cache is under about 2 GB, in addition to the raw images. These are planning bounds, not guaranteed minima.
Local Colab storage is ephemeral. Keep
or download the output receipts and completed records before ending a
runtime. To resume, restore the same cache/output folders and rerun in
order; the code verifies provenance before reusing completed work.


## 1. Repository setup

Clone the published `main` branch into `/content/noisy-anchor-alignment`.
An existing checkout is reused without deleting raw data or outputs.
For local Jupyter, change `REPO_ROOT` to the existing repository directory.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path("/content/noisy-anchor-alignment")
REPO_URL = 'https://github.com/KU-Nosaka/noisy-anchor-alignment.git'
REPO_REF = "main"

if not REPO_ROOT.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1", "--branch", REPO_REF,
        REPO_URL, str(REPO_ROOT),
    ])
else:
    print("Reusing existing repository directory:", REPO_ROOT)

API_ROOT = REPO_ROOT / "reproduction" / "casia"
if not (API_ROOT / "notebook_api.py").is_file():
    raise FileNotFoundError(
        "This checkout does not contain the current CASIA reproduction "
        "API. Use the updated repository main branch before continuing."
    )
sys.path.insert(0, str(API_ROOT))
print("Reproduction modules:", API_ROOT)


## 2. Install packages and check the runtime

Retain Colab's CUDA-enabled PyTorch and torchvision. Install the frozen
auditor package with `--no-deps` so its older dependency constraints do
not downgrade those GPU packages. No model is fine-tuned by this notebook.


In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "numpy", "scipy", "pandas", "pillow", "matplotlib",
    "scikit-learn", "tqdm", "requests", "gdown", "psutil",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-deps",
    "facenet-pytorch==2.6.0",
])

import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime with a Tesla T4 before running.")
GPU_NAME = torch.cuda.get_device_name(0)
if "T4" not in GPU_NAME:
    raise RuntimeError(f"Expected a Tesla T4; this runtime provides {GPU_NAME}.")
DEVICE = "cuda"
print("GPU:", GPU_NAME)
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)


## 3. Paths and fixed study settings

Only the raw input location needs to be supplied. All caches and new
outputs remain in this checkout. The constants below document the fixed
recipe implemented by the versioned reproduction API; the generated
configuration and receipts record the effective settings. To implement
a different experiment, change the versioned runner configuration rather
than only editing these descriptive constants.

Required entries relative to the raw dataset root: `img_align_celeba/*.jpg`, `list_attr_celeba.txt` (or the documented CSV variant), and `identity_CelebA.txt` (or its CSV variant).


In [ ]:
import json
import shutil
import psutil
import notebook_api as api

DATASET = 'celeba'
DATA_ROOT = REPO_ROOT / "data" / "raw" / DATASET
CACHE_ROOT = REPO_ROOT / "cache" / "current_casia" / DATASET
OUTPUT_ROOT = REPO_ROOT / "build" / "current_casia" / DATASET / 'attack_comparison500'
CONTEXT_ROOT = OUTPUT_ROOT / "context"

STUDY_SEED = 20260928
COHORT_SEED = 20260713
ATTACK_SEED = 20260930
FEATURE_DIMENSION = 400
ANCHOR_ROWS = 1600
PARTICIPANTS = [2, 5, 10, 20, 50]
IDENTITIES_PER_PARTICIPANT = 100
LINKAGE_BIN_EDGES = [0, 15, 25, 35, 45, 100]
OBSERVATIONS_PER_BIN = 20
CALIBRATION_PROPOSAL_CAP = 1500
EXPECTED_FITS_BY_P = {2: 204, 5: 207, 10: 212, 20: 222, 50: 252}
EXPECTED_FULL_FITS = 1097
ATTACK_DRAWS = 500
ANCHOR_NOISE_RANGE = (0.0, 0.08)
PRIVATE_NOISE_RANGE = (0.0, 3.0)

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"Put the raw {DATASET} images/archives and metadata inside "
        f"the repository at {DATA_ROOT} before continuing."
    )
for path in (CACHE_ROOT, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print("Raw data:", DATA_ROOT)
print("Preparation cache:", CACHE_ROOT)
print("New output:", OUTPUT_ROOT)
print("Free local disk (GiB):", round(shutil.disk_usage(REPO_ROOT).free / 2**30, 1))
print("Host RAM total / available (GiB):", round(psutil.virtual_memory().total / 2**30, 1),
      "/", round(psutil.virtual_memory().available / 2**30, 1))


## 4. Validate raw inputs, select identities, and preprocess images

The preparation stage validates raw metadata and image sources, excludes
reserved reference images from utility uploads, fixes the 50-participant
identity allocation, and builds RGB64 images in channel-first order.
CelebA uses centered square crops. VGGFace2 uses the supplied face boxes
expanded by 1.3, clipped to the image, and centered to a square.
VGGFace2 utility uses the released MAAD-Face Smiling labels; images with
unknown labels are excluded.

**This cell is always required.** A missing raw preparation cannot be
skipped merely because a later receipt or old output folder exists.
Verified caches may be reused, but a new run computes everything from
raw inputs. Progress messages identify the preparation and SVD stages.


In [ ]:
prepared_receipt = api.prepare_dataset(
    DATASET, data_root=DATA_ROOT, cache_root=CACHE_ROOT, device=DEVICE,
)
print(json.dumps(prepared_receipt, indent=2, default=str))


## 5. Public SVD feature mapping

The preceding preparation cell fits an **uncentered truncated SVD on
10,000 public images**, whose identities are disjoint from participant
identities. It produces a d = 12,288 by h = 400 orthonormal feature
basis, projects the participant/query/reference rows, and writes source
and feature receipts. An existing basis is reused only after the raw
source and configuration checks pass. No pretrained/private feature
file is supplied as an input.

The receipt printed above identifies the completed feature preparation.
The next stage verifies that prepared data before constructing the
frozen protocol context; a failed SVD stage prevents the experiment.


In [ ]:
expected_preparation = {
    "state": "prepared", "participant_count": 50, "identity_count": 5000,
    "public_svd_images": 10000, "h": 400,
}
for key, expected in expected_preparation.items():
    if prepared_receipt.get(key) != expected:
        raise RuntimeError(f"Preparation receipt mismatch for {key}: {prepared_receipt.get(key)}")
for key in ("manifest_path", "verification_path"):
    if not Path(prepared_receipt[key]).is_file():
        raise FileNotFoundError(f"Missing completed feature receipt: {prepared_receipt[key]}")
print("Public SVD and participant projections verified:", prepared_receipt["master_manifest_hash"])


## 6. Frozen CASIA auditor and protocol context

Load and freeze CASIA-WebFace InceptionResnetV1. The code verifies the
pretrained asset and runs the same RGB64 projection/decoding, 160-pixel
bilinear resize, standardization, and normalized cosine matching used
in the study. It does not train the auditor on either evaluation dataset.


In [ ]:
context = api.load_context(
    DATASET, data_root=DATA_ROOT, cache_root=CACHE_ROOT,
    context_root=CONTEXT_ROOT, p=10, device=DEVICE,
)
print("Frozen context is ready for", DATASET, "at p = 10.")


## 7. GDP, shared anchors, and identity galleries

The verified context fixes Haar orthogonal GDP matrices and Gaussian
translations for 50 participants, r = 1,600 centered orthonormal anchor
rows, and the same protected participant and genuine references. The
reference/colluding participant is participant 0; the protected one is
participant 1. C-GDP uses the reference participant's transformation. For
each collaboration size, 100 protected queries use ten ten-way galleries
each: 1,000 lineups with 10% random-guessing accuracy. Distractors come
from that active collaboration; lineups stay fixed across methods and
noise draws at the same p. Source/configuration hashes are retained in
the experiment records.


## 8. Run 500 uniform draws and all three attacks

Each draw generates one fresh noisy anchor upload for every active
participant. All three estimators attack that same upload, using
the same protected queries and galleries. No utility classifier is
trained. The seed and unconditional noise schedule are committed
before draw evaluation. A zero-noise diagnostic is saved separately.


In [ ]:
import attack_comparison as attack

terminal_summary = attack.run_attack_comparison(
    DATASET, data_root=DATA_ROOT, cache_root=CACHE_ROOT,
    output_root=OUTPUT_ROOT, device=DEVICE, seed=ATTACK_SEED,
)
print(json.dumps(terminal_summary, indent=2, default=str))


## 9. Inspect paired records and export the comparison

Full success requires exactly 500 unique random draws and all three
attack outcomes for every draw. Inspect the separate zero-noise
diagnostic, immutable draw receipts, and completed schedule before
interpreting the curves. The exported spline curves describe these
observations; their shaded 95% bootstrap bands resample draw indices
jointly across attacks, with the diagnostic excluded.


In [ ]:
verified_summary = attack.inspect_results(OUTPUT_ROOT)
print(json.dumps(verified_summary, indent=2, default=str))


In [ ]:
exported_files = attack.export_figures(OUTPUT_ROOT)
print(json.dumps(exported_files, indent=2, default=str))
from IPython.display import FileLink, Image, display

for figure_path in exported_files.values():
    path = Path(figure_path)
    if not path.is_file():
        raise FileNotFoundError(f"Exported artifact is missing: {path}")
    print(path)
    if path.suffix.lower() == ".png":
        display(Image(filename=str(path)))
    else:
        display(FileLink(str(path)))


## Retain the run

Keep the raw input hashes, preparation and configuration receipts, condition
records, calibration/schedule receipts, and terminal summary together. A
live progress message alone is not proof of completion. Resume only from
these verified folders; do not mix outputs from different seeds or settings.
